In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)
from typing import Iterable
try:
    from typing import Self
except ImportError:
    Self = object
class Rotation:
    def __init__(self, quat=None):
        self._quat = quat if quat is not None else np.zeros((1,4))
    def apply(self, v): return v
    def inv(self): return self
    def concatenate(self, other): return self



/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

_CSV_COLUMNS = ["z", "y", "x", "zvec", "yvec", "xvec"]

class Rotation:
    def __init__(self, quat=None):
        self._quat = np.asarray(quat) if quat is not None else np.zeros((1, 4))
    @classmethod
    def from_rotvec(cls, rotvec):
        arr = rotvec.to_numpy() if hasattr(rotvec, "to_numpy") else np.asarray(rotvec)
        quat = np.zeros((len(arr), 4))
        if len(arr):
            quat[:, -1] = 1.0
        return cls(quat)
    def apply(self, v): return v
    def inv(self): return self
    def concatenate(self, other): return self

class _MockMolecules:
    def __init__(self, pos, rot, features=None):
        self._pos = pos.to_numpy() if hasattr(pos, "to_numpy") else np.asarray(pos)
        self._rotator = rot
        self._features = features
        self.pos = self._pos
        self.features = features
    def quaternion(self):
        n = len(self.pos)
        quat = np.zeros((n, 4))
        if n:
            quat[:, -1] = 1.0
        return quat
    def rotvec(self):
        return np.zeros((len(self.pos), 3))
    def to_dataframe(self):
        data = np.concatenate([self.pos, self.rotvec()], axis=1)
        if isinstance(self._features, pl.DataFrame):
            df = pl.DataFrame(data, schema=_CSV_COLUMNS)
            return pl.concat([df, self._features], how="horizontal") if self._features is not None else df
        df = pd.DataFrame(data, columns=_CSV_COLUMNS)
        return pd.concat([df, self._features.reset_index(drop=True)], axis=1) if self._features is not None else df

MockRotation = Rotation
Molecules = _MockMolecules

FIX_POS_A = np.array([[1., 2., 3.], [4., 5., 6.]])
FIX_POS_B = np.array([[7., 8., 9.]])
FIX_QUAT_A = np.array([[0., 0., 0., 1.], [0., 0., 0., 1.]])
FIX_FEATURES_PD_A = pd.DataFrame({"score": [0.9, 0.8]})
FIX_FEATURES_PD_B = pd.DataFrame({"score": [0.7]})
FIX_FEATURES_PL_A = pl.from_pandas(FIX_FEATURES_PD_A)
FIX_FEATURES_PL_B = pl.from_pandas(FIX_FEATURES_PD_B)

_FEATURES_UNSET = object()
def _set_molecules_self_pd(features=_FEATURES_UNSET):
    global self
    self = _MockMolecules(
        np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]]),
        MockRotation(),
        pd.DataFrame({"score": [0.9, 0.8, 0.7]}) if features is _FEATURES_UNSET else features,
    )

def _set_molecules_self_pl(features=_FEATURES_UNSET):
    global self
    base = pl.DataFrame({"score": [0.9, 0.8, 0.7]}) if features is _FEATURES_UNSET else features
    self = _MockMolecules(np.array([[1., 2., 3.], [4., 5., 6.], [7., 8., 9.]]), MockRotation(), base)

# --- acryo_molecules_concat_migration ---
FIX_ACRYO_MOLECULES_CONCAT_MIGRATION_FEATURES = FIX_FEATURES_PD_A
FIX_ACRYO_MOLECULES_CONCAT_MIGRATION_POS = FIX_POS_A
FIX_ACRYO_MOLECULES_CONCAT_MIGRATION_QUAT = FIX_QUAT_A

# --- acryo_molecules_from_csv_migration ---
FIX_ACRYO_CSV_PATH = "acryo_molecules_fixture.csv"
pd.DataFrame({
    "z": [1.0, 4.0], "y": [2.0, 5.0], "x": [3.0, 6.0],
    "zvec": [0.0, 0.0], "yvec": [0.0, 0.0], "xvec": [0.0, 0.0],
    "score": [0.9, 0.8],
}).to_csv(FIX_ACRYO_CSV_PATH, index=False)

# --- acryo_molecules_rotation_copy_migration ---
FIX_ACRYO_MOLECULES_ROTATION_COPY_MIGRATION_ROT = MockRotation()

# --- acryo_molecules_subset_migration ---
FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_POS = np.array([[1., 2., 3.], [4., 5., 6.]])
FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_QUAT = np.array([[0., 0., 0., 1.], [0., 0., 0., 1.]])
FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_SPEC = [0, 1]

# --- acryo_molecules_to_csv_migration ---
FIX_ACRYO_TO_CSV_PD_PATH = "acryo_to_csv_pd.csv"
FIX_ACRYO_TO_CSV_PL_PATH = "acryo_to_csv_pl.csv"

# --- acryo_molecules_to_dataframe_migration ---

# --- acryo_molecules_translation_copy_migration ---
FIX_ACRYO_MOLECULES_TRANSLATION_COPY_MIGRATION_COORDS = np.array([[10., 20., 30.], [40., 50., 60.]])

_set_molecules_self_pd()
print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_acryo_molecules_concat_migration(features, pos, quat):
    @classmethod
    def concat(cls, moles: Iterable[Molecules], concat_features: bool = True) -> Self:
        """Concatenate Molecules objects."""
        pos: list[np.ndarray] = []
        quat: list[np.ndarray] = []
        features: list[pd.DataFrame] = []
        for mol in moles:
            pos.append(mol.pos)
            quat.append(mol.quaternion())
            features.append(mol.features)

        all_pos = np.concatenate(pos, axis=0)
        all_quat = np.concatenate(quat, axis=0)
        if concat_features:
            import pandas as pd

            all_features = pd.concat(features, axis=0)
        else:
            all_features = None

        return cls(all_pos, Rotation(all_quat), features=all_features)
    return concat

def before_acryo_molecules_from_csv_migration():
    def from_csv(
        cls,
        path: str,
        pos_cols: list[str] = ["z", "y", "x"],
        rot_cols: list[str] = ["zvec", "yvec", "xvec"],
        **pd_kwargs,
    ) -> Self:
        import pandas as pd
        pos_cols = pos_cols.copy()
        rot_cols = rot_cols.copy()
        df: pd.DataFrame = pd.read_csv(path, **pd_kwargs)  # type: ignore
        pos = df[pos_cols]
        rotvec = df[rot_cols]
        cols = pos + rotvec
        others = df.iloc[:, np.array([c not in cols for c in df.columns])]
        return cls(
            pos,
            Rotation.from_rotvec(rotvec),
            features=others,
        )
    return from_csv

def before_acryo_molecules_rotation_copy_migration(rot, features=None):
    if features is None:
        features = pd.DataFrame({"col":["a"],"val":[1.0]})
    if features is not None:
        features = features.copy()
    out = self.__class__(self._pos, rot, features=features)
    return out

def before_acryo_molecules_subset_migration(pos, quat, spec):
    if self._features is None:
        return self.__class__(pos, Rotation(quat))
    return self.__class__(pos, Rotation(quat), self._features.iloc[spec, :])
    return None

def before_acryo_molecules_to_csv_migration():
    def to_csv(self, save_path: str) -> None:
        """
        Save molecules as a csv file.

        Parameters
        ----------
        save_path : str
            Save path.
        """
        return self.to_dataframe().to_csv(save_path, index=False)
    return to_csv

def before_acryo_molecules_to_dataframe_migration():
    def to_dataframe(self) -> pd.DataFrame:
        import pandas as pd
        rotvec = self.rotvec()
        data = np.concatenate([self.pos, rotvec], axis=1)
        df = pd.DataFrame(data, columns=_CSV_COLUMNS)
        if self._features is not None:
            df = pd.concat([df, self._features], axis=1)
        return df
    return to_dataframe

def before_acryo_molecules_translation_copy_migration(coords, features=None):
    if features is None:
        features = pd.DataFrame({"col":["a"],"val":[1.0]})
    if features is not None:
        features = features.copy()
    out = self.__class__(coords, self._rotator, features=features)
    return out

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_acryo_molecules_concat_migration(features, pos, quat):
    import numpy as np
    from typing import Iterable, Self


    @classmethod
    def concat(cls, moles: Iterable[Molecules], concat_features: bool = True) -> Self:
        """Concatenate Molecules objects."""
        pos: list[np.ndarray] = []
        quat: list[np.ndarray] = []
        features: list[pl.DataFrame] = []
        for mol in moles:
            pos.append(mol.pos)
            quat.append(mol.quaternion())
            features.append(mol.features)

        all_pos = np.concatenate(pos, axis=0)
        all_quat = np.concatenate(quat, axis=0)
        if concat_features:
            all_features = pl.concat(features, how="vertical")
        else:
            all_features = None

        return cls(all_pos, Rotation(all_quat), features=all_features)
    return concat

def gen_acryo_molecules_from_csv_migration():
    def from_csv(
        cls,
        path: str,
        pos_cols: list[str] = ["z", "y", "x"],
        rot_cols: list[str] = ["zvec", "yvec", "xvec"],
        **pd_kwargs,
    ) -> Self:

        pos_cols = pos_cols.copy()
        rot_cols = rot_cols.copy()
        df: pl.DataFrame = pl.read_csv(path, **pd_kwargs)
        pos = df.select(pos_cols)
        rotvec = df.select(rot_cols)
        cols = pos_cols + rot_cols
        others = df.select([c for c in df.columns if c not in cols])
        return cls(
            pos,
            Rotation.from_rotvec(rotvec),
            features=others,
        )
    return from_csv

def gen_acryo_molecules_rotation_copy_migration(rot, features=None):
    if features is None:
        features = pl.DataFrame({"col":["a"],"val":[1.0]})
    if features is not None:
        features = features.clone() if hasattr(features, "clone") else features.copy()
    out = self.__class__(self._pos, rot, features=features)
    return out

def gen_acryo_molecules_subset_migration(pos, quat, spec):
    if self._features is None:
        return self.__class__(pos, Rotation(quat))
    if self._features is None:
        return self.__class__(pos, Rotation(quat))
    return self.__class__(pos, Rotation(quat), self._features[spec, :])

def gen_acryo_molecules_to_csv_migration():
    def to_csv(self, save_path: str) -> None:
        """
        Save molecules as a csv file.

        Parameters
        ----------
        save_path : str
            Save path.
        """
        return self.to_dataframe().write_csv(save_path)
    return to_csv

def gen_acryo_molecules_to_dataframe_migration():
    import numpy as np

    def to_dataframe(self) -> pl.DataFrame:
        rotvec = self.rotvec()
        data = np.concatenate([self.pos, rotvec], axis=1)
        df = pl.DataFrame(data, schema=_CSV_COLUMNS)
        if self._features is not None:
            df = pl.concat([df, pl.DataFrame(self._features)], how="horizontal")
        return df
    return to_dataframe

def gen_acryo_molecules_translation_copy_migration(coords, features=None):
    if features is None:
        features = pl.DataFrame({"col":["a"],"val":[1.0]})
    if features is not None:
        features = features.clone()
    out = self.__class__(coords, self._rotator, features=features)
    return out

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:
# === Tests: acryo_molecules_subset_migration ===

try:
    _set_molecules_self_pl()
    _r = gen_acryo_molecules_subset_migration(FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_POS, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_QUAT, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_SPEC)
    print("✅ L1 smoke gen_acryo_molecules_subset_migration: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_acryo_molecules_subset_migration: {type(_e).__name__}: {_e}")

try:
    _set_molecules_self_pd()
    _rb = before_acryo_molecules_subset_migration(FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_POS, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_QUAT, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_SPEC)
    print("✅ L1 smoke before_acryo_molecules_subset_migration: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_acryo_molecules_subset_migration: {type(_e).__name__}: {_e}")

try:
    _set_molecules_self_pd()
    _rb = before_acryo_molecules_subset_migration(FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_POS, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_QUAT, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_SPEC)
    _set_molecules_self_pl()
    _rg = gen_acryo_molecules_subset_migration(FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_POS, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_QUAT, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_SPEC)
    compare(_rb.features, _rg.features, "acryo_molecules_subset_migration features", check_row_order=True)
except Exception as _e:
    print(f"❌ L2 equivalence acryo_molecules_subset_migration: setup error — {type(_e).__name__}: {_e}")

# AUDIT-160: compare the no-features branch on both sides.
try:
    _set_molecules_self_pd(features=None)
    _rb = before_acryo_molecules_subset_migration(FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_POS, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_QUAT, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_SPEC)
    _set_molecules_self_pl(features=None)
    _rg = gen_acryo_molecules_subset_migration(FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_POS, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_QUAT, FIX_ACRYO_MOLECULES_SUBSET_MIGRATION_SPEC)
    if _rb.features is None and _rg.features is None and np.array_equal(_rb.pos, _rg.pos):
        print("✅ L3 edge acryo_molecules_subset_migration no features oracle: MATCH")
    else:
        print("❌ L3 edge acryo_molecules_subset_migration no features oracle: MISMATCH")
except Exception as _e:
    print(f"❌ L3 edge acryo_molecules_subset_migration: {type(_e).__name__}: {_e}")


✅ L1 smoke gen_acryo_molecules_subset_migration: OK, type= _MockMolecules
✅ L1 smoke before_acryo_molecules_subset_migration: OK
✅ L2 equivalence acryo_molecules_subset_migration features: MATCH
✅ L3 edge acryo_molecules_subset_migration no features oracle: MATCH
